# Actividad 5: Construcción de la base de datos

Este cuaderno une cinco series en **una sola base** y escribe los dos archivos que usan los modelos de
nowcasting de las próximas sesiones:

* **`data/data.csv`**: una fila por mes, una columna por serie, cada columna con su *ticker*;
* **`data/meta.csv`**: una fila por serie, que dice qué es, en qué unidades está, si es nominal,
  si ya viene desestacionalizada, cómo se agrega y qué tan atrasada llega.

| archivo en `data/` | serie | frecuencia |
|---|---|---|
| `gdp.csv` | PIB real, volumen encadenado ref. 2017 (INE) | trimestral |
| `ntl.csv` | luces nocturnas, radiancia media de Bolivia (NASA Black Marble) | diaria |
| `gtrends.csv` | Google Trends, promedio de los términos de un sector («transporte») | mensual |
| `ine_ipc.csv` | índice de precios al consumidor (INE) | mensual |
| `ine_importaciones_destino_economico.csv` | importaciones totales, valor (INE) | mensual |

La segunda mitad usa la base **a través de sus metadatos**: deflacta lo nominal, desestacionaliza,
pasa a trimestral, calcula tasas de crecimiento y estima un OLS contra el PIB.

Las celdas con `<SU CÓDIGO AQUÍ: ...>` son para completar. La solución está en `sols/act5_build.ipynb`.

In [ ]:
# ===========================================================================
# CARGA DE LA CONFIGURACIÓN  --  editar config.py, NO esta celda.
# Deja `cfg` (los parámetros del país) y `P` (las rutas estándar).
# ===========================================================================
import os, sys, importlib.util

CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

PATH_DATA = P["data"]     # aquí están los insumos y aquí se escriben data.csv y meta.csv

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})")
print(f"         data : {PATH_DATA}")
print(f"         PIB = {cfg.GDP_TICKER}   IPC = {cfg.CPI_TICKER}")

In [ ]:
# ===========================================================================
# Librerías y funciones de apoyo.
# ===========================================================================
import re, unicodedata, warnings
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings("ignore")
HOY = pd.Timestamp(date.today())


def frecuencia(indice):
    # Deduce la frecuencia a partir de la distancia típica (mediana) entre fechas.
    dias = pd.Series(pd.DatetimeIndex(indice).sort_values()).diff().dt.days.median()
    if dias <= 2:    return "diaria"
    if dias <= 10:   return "semanal"
    if dias <= 35:   return "mensual"
    if dias <= 100:  return "trimestral"
    return "anual"


print("listo")

---
# Parte 1: Metadata

## Paso 1. Manual

Hay información que **ningún programa puede adivinar** mirando los números: qué es la serie, sus unidades,
si está en precios corrientes, si ya viene desestacionalizada y cómo se debe agregar a una frecuencia más baja.
Eso se escribe a mano, **una vez**, en una tabla.

| campo | qué es |
|---|---|
| `file`, `series` | de qué archivo y qué columna sale la serie: el vínculo con los datos crudos |
| `ticket` | el nombre corto con el que la serie entra en la base: `<bloque><nemónico><número>` |
| `nominal` | `1` si está en precios corrientes (hay que deflactarla), `0` si no |
| `sa` | `1` si ya viene desestacionalizada, `0` si no |
| `agg_method` | cómo se pasa de una frecuencia alta a una baja: `mean` (promedio), `last` (último valor), `sum` (suma) |

Bloques del *ticker*: **R** actividad real · **M** dinero y precios · **X** sector externo ·
**G** Google Trends · **U** satélite. El PIB y el IPC tienen el *ticker* fijado en `config.py`
(`RGDP0000`, `MCPI0000`) porque los modelos los buscan por ese nombre.

In [ ]:
# Google Trends: se usa el promedio de los términos de UN sector.
# Los sectores están en la hoja "keywords" de BOL_GTrends_Keywords (columna "sector").
SECTOR_GT = "transporte"


def a_columna(termino):
    # 'pasajes aéreos' -> 'gt_pasajes_aereos', el nombre de la columna en gtrends.csv
    t = unicodedata.normalize("NFKD", termino).encode("ascii", "ignore").decode().lower()
    return "gt_" + re.sub(r"[^a-z0-9]+", "_", t).strip("_")


claves = pd.read_excel(os.path.join(PATH_DATA, "BOL_GTrends_Keywords_2026-09-18.xlsx"), sheet_name="keywords")
descargadas = pd.read_csv(os.path.join(PATH_DATA, "gtrends.csv"), nrows=0).columns
TERMINOS_GT = sorted({a_columna(k) for k in claves.loc[claves["sector"] == SECTOR_GT, "keyword"]} & set(descargadas))
assert TERMINOS_GT, f"ningún término del sector {SECTOR_GT} está en gtrends.csv"
print(f"Google Trends, sector {SECTOR_GT}: {TERMINOS_GT}")
print("sectores disponibles:", sorted(claves["sector"].unique()))

FUENTES = [
    dict(file="gdp.csv", series="gdp", filtro=None,
         ticket=cfg.GDP_TICKER, units="millones de Bs encadenados (ref. 2017)",
         description="PIB real trimestral, volumen encadenado (INE)",
         nominal=0, sa=0, agg_method="last"),
    dict(file="ine_ipc.csv", series="ipc_indice_general", filtro=None,
         ticket=cfg.CPI_TICKER, units="índice 2016 = 100",
         description="Índice de precios al consumidor, general (INE)",
         nominal=0, sa=0, agg_method="mean"),
    dict(file="ine_importaciones_destino_economico.csv", series="importaciones__total", filtro=None,
         ticket="XIMP0001", units="millones de USD, precios corrientes",
         description="Importaciones totales (INE)",
         <SU CÓDIGO AQUÍ: nominal=<¿1 o 0?>, sa=0, agg_method=<¿mean, last o sum?>>),
    dict(file="gtrends.csv", series="promedio", filtro=None,
         ticket="GGTR0001", units="índice 0-100, promedio de los términos",
         description=f"Google Trends, Bolivia: promedio del sector {SECTOR_GT}",
         nominal=0, sa=0, agg_method="mean"),
    dict(file="ntl.csv", series="ntl_mean", filtro=None,
         ticket="UNTL0001", units="nW/cm²/sr",
         description="Luces nocturnas: radiancia media de Bolivia (VNP46A2)",
         nominal=0, sa=0, agg_method="mean"),
]

fuentes = pd.DataFrame(FUENTES)
assert fuentes["ticket"].is_unique, "dos series con el mismo ticket"
fuentes[["ticket", "file", "series", "nominal", "sa", "agg_method"]]

## Paso 2. Descripcion de serie en metadata

Para cada fila de la tabla: abrir el archivo, tomar la columna y anotar:
frecuencia, primera y última fecha, número de observaciones.

`filtro` deja aplicar una condición antes de tomar la columna.

`gtrends.csv` trae una columna por término (65). La serie de Google Trends es el **promedio de los términos de un
sector**, elegido con `SECTOR_GT` en la celda anterior. Cada término por separado es ruidoso; el promedio lo suaviza.

¿Por qué no el promedio de **todos** los términos? Porque se mezclan búsquedas que se mueven en sentidos opuestos.
Las de consumo (*delivery*, *celular*, *laptop*) crecieron sin pausa desde 2018 mientras el PIB se desaceleraba, y
cancelan a las que sí siguen la actividad. El promedio de los 65 términos tiene correlación **negativa** con el
crecimiento del PIB. El sector **transporte** (*boa*, *pasajes*) refleja la demanda de viajes, que cae y se
recupera con la actividad, igual que las luces de los aeropuertos.

In [ ]:
crudas, descripcion = {}, []

for f in FUENTES:
    df = pd.read_csv(os.path.join(PATH_DATA, f["file"]), index_col="date", parse_dates=True)
    if f["filtro"]:
        df = df.query(f["filtro"])
    if f["series"] == "promedio":                       # Google Trends: promedio de los términos del sector
        df["promedio"] = df[TERMINOS_GT].mean(axis=1)
    s = pd.to_numeric(df[f["series"]], errors="coerce").dropna().sort_index()
    crudas[f["ticket"]] = s
    descripcion.append(dict(ticket=f["ticket"], freq_raw=frecuencia(s.index),
                            start_date=s.index.min(), end_date=s.index.max(), obs=len(s)))

descripcion = pd.DataFrame(descripcion)
descripcion

## Paso 3. Remuestro

La base es **mensual**.

* **diaria** (luces): se promedian los días de cada mes, salvo el mes en curso, que está incompleto;
* **mensual**: se fecha el primer día del mes;
* **trimestral** (PIB): queda en el **último mes** del trimestre (2025T4 = 2025-12-01) y los otros dos meses
  quedan vacíos. Así cada dato se ubica en el mes en que termina el periodo que mide.

In [ ]:
MES_EN_CURSO = HOY.to_period("M").to_timestamp()


def a_mensual(s, freq_raw):
    if freq_raw in ("diaria", "semanal"):
        s = s[s.index < MES_EN_CURSO]           # el mes en curso todavía no terminó
        return s.resample("MS").mean()
    # mensual o trimestral: sólo se lleva cada fecha al día 1 de su mes
    return s.groupby(s.index.to_period("M").to_timestamp()).last()


frec = descripcion.set_index("ticket")["freq_raw"]
mensuales = {t: a_mensual(s, frec[t]) for t, s in crudas.items()}

fig, ax = plt.subplots(figsize=(11, 3.8))
crudas["UNTL0001"].loc["2024":].plot(ax=ax, ls="", marker=".", ms=2, alpha=.4, label="diaria")
mensuales["UNTL0001"].loc["2024":].plot(ax=ax, lw=2.2, color="darkorange", label="promedio mensual")
ax.set_title("Luces nocturnas: de diaria a mensual"); ax.legend()
plt.show()

## Paso 4. Combinar y escribir `data.csv`

Una columna por *ticker*, el PIB primero. La base empieza **dos meses antes** del primer dato del PIB
(el primer trimestre completo: enero, febrero y marzo de 2017).

Mire el final de la tabla: cada serie termina en un mes distinto. Ese **borde irregular** (*ragged edge*) es
justamente lo que el nowcast aprovecha: las series mensuales ya informan sobre trimestres cuyo PIB todavía no
se publicó.

In [ ]:
data = pd.DataFrame(mensuales)
data.index.name = "date"
data = data[[cfg.GDP_TICKER] + [c for c in data.columns if c != cfg.GDP_TICKER]]   # el PIB primero

inicio = data[cfg.GDP_TICKER].first_valid_index() - pd.DateOffset(months=2)
data = data.loc[inicio:]

data.to_csv(os.path.join(PATH_DATA, cfg.DATA_CSV))
print(f"data.csv: {data.shape[0]} meses x {data.shape[1]} series, "
      f"{data.index.min().date()} a {data.index.max().date()}")
data.tail(12).round(1)

## Paso 5. Finalizar metadata: escribir `meta.csv`

`meta.csv` combina lo escrito a mano (paso 1) y la descripción calculada (paso 2), más un dato clave:

* **`months_lag`**: cuántos meses pasaron entre el último dato y hoy, es decir, el **rezago de publicación**.
  Se recalcula en cada corrida.
* **`freq`**: la frecuencia **con que la serie entra en la base** (`quarterly` o `monthly`); la original queda
  en `freq_raw`.

`meta.csv` es un contrato con los datos:
si se cambia el nombre de una columna en un archivo crudo,
la fila deja de encontrarla y la serie desaparece de la base.

In [ ]:
meta = fuentes.drop(columns="filtro").merge(descripcion, on="ticket")

meta["category"] = meta["ticket"].str[0]
meta["freq"] = np.where(meta["freq_raw"] == "trimestral", "quarterly", "monthly")
meta["months_lag"] = [(HOY.year - d.year) * 12 + (HOY.month - d.month)
                      for d in meta["end_date"]]
for c in ("start_date", "end_date"):
    meta[c] = meta[c].dt.strftime("%Y-%m-%d")

COLUMNAS = ["file", "series", "ticket", "category", "units", "description", "nominal", "sa",
            "agg_method", "freq", "freq_raw", "start_date", "end_date", "obs", "months_lag"]
meta = meta[COLUMNAS]
meta.to_csv(os.path.join(PATH_DATA, cfg.METADATA_CSV), index=False)

print(f"meta.csv: {len(meta)} series")
meta

---
# Parte 2: usar la data a través de sus metadatos

De aquí en adelante el cuaderno **sólo lee `data.csv` y `meta.csv`**, como lo harán los modelos. Ninguna
decisión se toma por el nombre de una serie: cada paso consulta su columna de metadatos.

| paso | columna de `meta.csv` |
|---|---|
| deflactar | `nominal` |
| desestacionalizar | `sa` |
| pasar a trimestral | `agg_method` |

In [ ]:
data = pd.read_csv(os.path.join(PATH_DATA, cfg.DATA_CSV), index_col="date", parse_dates=True)
meta = pd.read_csv(os.path.join(PATH_DATA, cfg.METADATA_CSV)).set_index("ticket")
print(data.shape, "|", len(meta), "filas de metadatos")
print(data.tail(5))

## Paso 6 · Deflactar

Una serie en **precios corrientes** sube con la inflación aunque no cambie la cantidad. Para llevarla a términos
reales se divide por el IPC, expresado con un año base = 100:

$$ \text{real}_t = \frac{\text{nominal}_t}{\text{IPC}_t} \times 100 $$

El año base es **2017**, el mismo de las cuentas nacionales: así las series reales quedan en precios de 2017,
como el PIB. Las importaciones están en dólares; deflactarlas con el IPC boliviano es una simplificación, y es
la misma regla que aplican los modelos a toda serie con `nominal == 1`.

In [ ]:
ANIO_BASE = 2017

ipc = data[cfg.CPI_TICKER]
ipc_base = ipc / ipc.loc[str(ANIO_BASE)].mean() * 100     # promedio de 2017 = 100

nominales = meta.index[meta["nominal"] == 1].tolist()
print("se deflactan:", nominales)

real = data.copy()
for t in nominales:
    real[t] = <SU CÓDIGO AQUÍ: la serie nominal dividida por ipc_base, por 100>

t = nominales[0]
fig, ax = plt.subplots(figsize=(11, 3.8))
data[t].plot(ax=ax, alpha=.6, label="nominal")
real[t].plot(ax=ax, lw=1.8, label=f"real (precios de {ANIO_BASE})")
ax.set_title(f"{meta.loc[t, 'description']}: nominal y real"); ax.legend()
plt.show()

## Paso 7 · Desestacionalizar

`seasonal_decompose` separa la serie en
tendencia + estacionalidad + resto; la serie desestacionalizada es la original **menos** la estacionalidad.

$$Y_t = \text{Trend}_t + \text{Seasonal}_t + \text{Residual}_t$$

El ajuste sigue la substracción de este componente:

$$\text{Adjusted}_t = Y_t - \text{Seasonal}_t = \text{Trend}_t + \text{Residual}_t$$


Se aplica a las series con `sa == 0`, con periodo 12 (mensual) o 4 (trimestral). Es un método simple,
suficiente para el taller. Las oficinas de estadística usan X-13 ARIMA-SEATS.

`seasonal_decompose()` toma estos argumentos

- **`model='additive'`**: asume los componentes son aditivos en vez de multiplicativos.
- **`period=12`**: es el ciclo o frecuencia
- **`extrapolate_trend='freq'`**: estima la tendencia con una media móvil. La primera observacón va a ser un `NaN` y se llena con una extrapolación lineal.

In [ ]:
t = 'UNTL0001'
d = seasonal_decompose(real[t], model="additive", period=12, extrapolate_trend="freq")
d = (real[t] - d.seasonal).reindex(real.index)

fig, ax = plt.subplots(figsize=(11, 3.8))
real[t].dropna().plot(ax=ax, alpha=.5, label="real")
d.dropna().plot(ax=ax, lw=1.8, label="real, desestacionalizada")
ax.set_title(meta.loc[t, "description"]); ax.legend()
plt.show()

Datos completos:

In [ ]:
desest = real.copy()

for t in meta.index[meta["sa"] == 0]:
    if meta.loc[t, "freq"] == "quarterly":
        s, periodo = real[t].dropna(), 4
    else:
        s, periodo = real[t].dropna(), <SU CÓDIGO AQUÍ: ¿periodo para una serie mensual?>
    d = seasonal_decompose(s, model="additive", period=periodo, extrapolate_trend="freq")
    desest[t] = (s - d.seasonal).reindex(real.index)

t = nominales[0]
fig, ax = plt.subplots(figsize=(11, 3.8))
real[t].dropna().plot(ax=ax, alpha=.5, label="real")
desest[t].dropna().plot(ax=ax, lw=1.8, label="real, desestacionalizada")
ax.set_title(meta.loc[t, "description"]); ax.legend()
plt.show()

Grafica las series de PIB, luces nocturnas, y Google Trends:

In [ ]:
t = "RGDP0000"
fig, ax = plt.subplots(figsize=(11, 3.8))
real[t].dropna().plot(ax=ax, alpha=.5, label="real")
desest[t].dropna().plot(ax=ax, lw=1.8, label="real, desestacionalizada")
ax.set_title(meta.loc[t, "description"]); ax.legend()
plt.show()

In [ ]:
t = "UNTL0001"
fig, ax = plt.subplots(figsize=(11, 3.8))
real[t].dropna().plot(ax=ax, alpha=.5, label="real")
desest[t].dropna().plot(ax=ax, lw=1.8, label="real, desestacionalizada")
ax.set_title(meta.loc[t, "description"]); ax.legend()
plt.show()

In [ ]:
t = "GGTR0001"
fig, ax = plt.subplots(figsize=(11, 3.8))
real[t].dropna().plot(ax=ax, alpha=.5, label="real")
desest[t].dropna().plot(ax=ax, lw=1.8, label="real, desestacionalizada")
ax.set_title(meta.loc[t, "description"]); ax.legend()
plt.show()

## Paso 8. Remuestreo trimestral

El PIB es trimestral. En este ejercicio llevaremos las series mensuales a trimestre. Cada una
según su `agg_method`: un flujo (importaciones) se **suma**; un índice (IPC, Google Trends, luces) se **promedia**.

Sólo se usan trimestres **completos**: un trimestre con un solo mes publicado no se puede comparar con uno de tres.

**Nota:** el nowcast sigue otro procedimiento.

In [ ]:
def a_trimestral(df):
    out = {}
    for t in df.columns:
        trim = df[t].resample("QS")
        n = trim.count()
        valor = trim.agg(meta.loc[t, "agg_method"])
        if meta.loc[t, "freq"] == "monthly":
            valor = valor.where(n == 3)                     # sólo trimestres completos
        out[t] = valor
    q = pd.DataFrame(out)
    q.index = q.index + pd.DateOffset(months=2)             # fechado en el último mes del trimestre
    return q


q_nsa = a_trimestral(real)      # sin desestacionalizar
q_sa  = a_trimestral(desest)    # desestacionalizada
q_nsa.tail(6).round(1)

## Paso 9. Tasas de crecimiento

Dos maneras de medir el crecimiento trimestral:

* **interanual (y/y)**: contra el mismo trimestre del año anterior, `pct_change(4)`. La estacionalidad se cancela
  sola, por eso funciona con la serie **sin** desestacionalizar;
* **trimestral (q/q)**: contra el trimestre anterior, `pct_change(1)`. Sólo tiene sentido con la serie
  **desestacionalizada**; si no, mide el patrón estacional.

**Cuidado:** `pct_change` rellena los vacíos con el último dato antes de calcular, e inventa tasas para
trimestres que todavía no se publicaron. `fill_method=None` lo evita.

El gráfico compara las dos para el PIB.

**El OLS del paso 10 usa la interanual**, sobre las series **sin** desestacionalizar. Es la tasa con que el INE y el
BCB reportan el crecimiento y la que predicen los modelos de nowcasting. Y como compara cada trimestre con el
mismo trimestre del año anterior, no necesita la desestacionalización del paso 7: ésa hace falta para la
trimestral, que se usa en el ejercicio final.

In [ ]:
yoy = <SU CÓDIGO AQUÍ: variación interanual en %, sobre q_nsa (con fill_method=None)>
qoq = q_sa.pct_change(1, fill_method=None) * 100 

fig, ax = plt.subplots(figsize=(11, 3.8))
yoy[cfg.GDP_TICKER].plot(ax=ax, lw=2, marker="o", ms=3, label="interanual (y/y), sin desestacionalizar")
qoq[cfg.GDP_TICKER].plot(ax=ax, lw=1.5, marker="o", ms=3, label="trimestral (q/q), desestacionalizada")
ax.axhline(0, color="k", lw=.8)
ax.set_title("PIB real: dos tasas de crecimiento, %"); ax.legend()
plt.show()

## Paso 10. Un OLS básico contra el PIB

$$ \Delta_{4}\text{PIB}_t = \beta_0 + \beta_1\,\Delta_{4}\text{Importaciones}_t + \beta_2\,\Delta_{4}\text{Trends}_t
   + \beta_3\,\Delta_{4}\text{Luces}_t + \varepsilon_t $$

Todas las series en variación interanual (%), es decir, las columnas de `yoy` del paso 9. La muestra la fija la
serie más corta: Google Trends empieza en 2018, así que su primera tasa interanual es de 2019.

### Cómo se estima con `statsmodels`

La regresión se arma en cuatro pasos:

1. **Elegir las variables.** Una lista `X_COLS` con los *tickers* explicativos: `"XIMP0001"` (importaciones),
   `"GGTR0001"` (Google Trends) y `"UNTL0001"` (luces). El PIB es `cfg.GDP_TICKER`.
2. **Armar la muestra.** Tomar de `yoy` el PIB y las explicativas, y quitar con `.dropna()` todo trimestre al que
   le falte **alguna** de ellas. OLS no acepta valores vacíos, y así todas las series usan los mismos trimestres.
3. **Separar `y` y `X`.**
   * `y` es la columna del PIB de la muestra.
   * `X` son las columnas explicativas, pasadas por `sm.add_constant(...)`. Esta función agrega una columna de
     unos llamada `const`: es la que estima $\beta_0$. **Sin ella, la recta queda obligada a pasar por cero.**
4. **Estimar.** `sm.OLS(y, X)` define el modelo (primero la dependiente, después las explicativas) y `.fit()`
   lo estima. El resultado, que aquí se llama `modelo`, guarda todo:

| atributo | qué contiene |
|---|---|
| `modelo.params` | los coeficientes $\beta$ |
| `modelo.pvalues` | los p-valores: < 0,05 se lee como significativo al 5 % |
| `modelo.rsquared` | el R²: qué parte de la variación del PIB explica el modelo |
| `modelo.fittedvalues` | el PIB ajustado, $\hat y$, un valor por trimestre |
| `modelo.resid` | los residuos, $y - \hat y$ |
| `modelo.summary()` | la tabla completa; `.tables[1]` es sólo la de coeficientes |

**Tarea.** Escriba la celda completa: estime el modelo, imprima la muestra usada (primer y último trimestre,
número de trimestres), la tabla de coeficientes y el R². La celda siguiente usa `X_COLS`, `muestra`, `y`
y `modelo`, así que use esos nombres.

In [ ]:
# <SU CÓDIGO AQUÍ: estime el OLS interanual siguiendo los cuatro pasos de arriba.
#   Use los nombres X_COLS, muestra, y, modelo; imprima la muestra, la tabla de coeficientes y el R²>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(y.index, y, color="#003865", linewidth=2,
        marker="o", markersize=4, label="PIB observado")
# modelo.fittedvalues: los valores ajustados (y sombrero) de la regresión
ax.plot(modelo.fittedvalues.index, modelo.fittedvalues,
        color="#003865", linewidth=1.5, linestyle="--",
        alpha=0.7, label="Ajustado")
# fill_between() sombrea el área entre dos líneas: aquí, la distancia entre observado y ajustado (el residuo)
ax.fill_between(y.index, y, modelo.fittedvalues,
                alpha=0.12, color="#003865", label="Residuo")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.set_title("PIB real, variación interanual: observado y ajustado", fontsize=12, weight="bold")
ax.set_ylabel("variación, %")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

# ¿Cuánto del ajuste se debe a la pandemia?
sin_covid = muestra.drop(muestra.loc["2020-04-01":"2021-06-30"].index)
m2 = sm.OLS(sin_covid[cfg.GDP_TICKER], sm.add_constant(sin_covid[X_COLS])).fit()
print(f"R² con 2020-21: {modelo.rsquared:.2f}   |   R² sin 2020T2-2021T2: {m2.rsquared:.2f} "
      f"({len(sin_covid)} trimestres)")

### Ejercicio: el mismo OLS con la tasa trimestral

Estime el mismo modelo con la variación **trimestral (q/q)** de las series **desestacionalizadas**, es decir,
con `qoq` en lugar de `yoy`:

$$ \Delta\text{PIB}_t = \beta_0 + \beta_1\,\Delta\text{Importaciones}_t + \beta_2\,\Delta\text{Trends}_t
   + \beta_3\,\Delta\text{Luces}_t + \varepsilon_t $$

Compare con el modelo interanual: el R², con y sin la pandemia, y qué coeficientes siguen siendo significativos.

**Para pensar.** Dos tasas interanuales seguidas comparten tres de sus cuatro trimestres, así que la serie
interanual es suave por construcción. ¿Cuánto del R² interanual es capacidad de predecir y cuánto es esa suavidad?

In [ ]:
# <SU CÓDIGO AQUÍ: el mismo OLS con qoq en lugar de yoy.
#   Compare el R², con y sin la pandemia, con el del modelo interanual>